In [49]:
## get working directory
import os
### Change working directory
os.chdir('D://BSE Semester 2//Introduction to Text Mining and NLP//Assignments and Exercises//Assign 1//DID Model//DID Model Data')
print(os.getcwd())

D:\BSE Semester 2\Introduction to Text Mining and NLP\Assignments and Exercises\Assign 1\DID Model\DID Model Data


In [11]:
# !pip install vaderSentiment
# !pip install textstat

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 3.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat

In [85]:
# Load the dataset
file_path = "D:/BSE Semester 2/Introduction to Text Mining and NLP/Assignments and Exercises/Assign 1/DID Model/DID Model Data/tfidf_vectorized_text.csv"
df = pd.read_csv(file_path)

# Preview data structure
df.head()

,city,hotel_name,price_euros_first,price_euros_second,rating,num_reviews,quality_over_5,amen,apartamento,art,...,shower,site,smart,sofa,somorrostro,spa,storag,theatr,tivoli,use
0,"Madrid, Community of Madrid",#10OCA6 - Apartamento en Vista Alegre,601.0,601.0,6.9,22.0,3.0,0.000000,3.380703,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Madrid, Community of Madrid",1881 Madrid Ventas Hotel,1000.0,1139.0,8.9,1395.0,4.0,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Madrid, Community of Madrid",2 bedrooms 1 bathroom furnished - Las Letras -...,1099.0,1658.0,7.5,28.0,3.0,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Madrid, Community of Madrid",4C Bravo Murillo,705.0,800.0,8.2,5764.0,2.0,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Madrid, Community of Madrid",5 Calle Álamo,2245.0,2245.0,NaN,NaN,NaN,3.786168,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
df.dtypes

city                   object
hotel_name             object
price_euros_first     float64
price_euros_second    float64
rating                float64
                       ...   
spa                   float64
storag                float64
theatr                float64
tivoli                float64
use                   float64
Length: 225, dtype: object

In [87]:
# Extract necessary columns
hotel_name_col = "hotel_name"
city_col = "city"

# List of columns to exclude (non-TFIDF columns)
exclude_columns = ['city', 'hotel_name', 'price_euros_first', 'price_euros_second', 'rating', 'num_reviews', 'quality_over_5']

# Select only the TF-IDF columns (all float64 columns except excluded ones)
tfidf_columns_select = [col for col in df.columns if col not in exclude_columns]

# Replace 0 values with NaN to exclude them from the mean calculation
df[tfidf_columns_select] = df[tfidf_columns_select].replace(0, np.nan)

In [89]:
### --- Compute TF-IDF Based Features --- ###

# List of columns to exclude (non-TFIDF columns)
exclude_columns = ['city', 'hotel_name', 'price_euros_first', 'price_euros_second', 'rating', 'num_reviews', 'quality_over_5']

# Select only the TF-IDF columns (all float64 columns except excluded ones)
tfidf_columns_select = [col for col in df.columns if col not in exclude_columns]

# Replace 0 values with NaN to exclude them from the mean calculation
df[tfidf_columns_select] = df[tfidf_columns_select].replace(0, np.nan)

# Compute the average, max, min, and std deviation of TF-IDF scores for each hotel
hotel_tfidf_stats = df.groupby("hotel_name")[tfidf_columns_select].agg(['mean', 'max', 'min', 'std']).reset_index()

# Flatten multi-index columns
hotel_tfidf_stats.columns = ['_'.join(col).strip('_') for col in hotel_tfidf_stats.columns]

# Rename columns for clarity
hotel_tfidf_stats.rename(columns={
    "hotel_name_": "hotel_name",
    "mean": "avg_tfidf_hotel",
    "max": "max_tfidf_hotel",
    "min": "min_tfidf_hotel",
    "std": "std_tfidf_hotel"
}, inplace=True)

# Compute overall summary statistics per hotel (excluding NaN values)
hotel_tfidf_stats["avg_tfidf_hotel"] = hotel_tfidf_stats[[col for col in hotel_tfidf_stats.columns if 'mean' in col]].mean(axis=1, skipna=True)
hotel_tfidf_stats["max_tfidf_hotel"] = hotel_tfidf_stats[[col for col in hotel_tfidf_stats.columns if 'max' in col]].max(axis=1, skipna=True)
hotel_tfidf_stats["min_tfidf_hotel"] = hotel_tfidf_stats[[col for col in hotel_tfidf_stats.columns if 'min' in col]].min(axis=1, skipna=True)
hotel_tfidf_stats["std_tfidf_hotel"] = hotel_tfidf_stats[[col for col in hotel_tfidf_stats.columns if 'std' in col]].mean(axis=1, skipna=True)

# Merge computed statistics back into the original dataframe
df = df.merge(hotel_tfidf_stats[['hotel_name', 'avg_tfidf_hotel', 'max_tfidf_hotel', 'min_tfidf_hotel', 'std_tfidf_hotel']], on="hotel_name", how="left")


In [93]:
# Display the first few rows
df[['hotel_name', 'avg_tfidf_hotel', 'max_tfidf_hotel', 'min_tfidf_hotel', 'std_tfidf_hotel' ]].head()

,hotel_name,avg_tfidf_hotel,max_tfidf_hotel,min_tfidf_hotel,std_tfidf_hotel
0,#10OCA6 - Apartamento en Vista Alegre,2.789024,3.956067,2.214951,NaN
1,1881 Madrid Ventas Hotel,3.073839,3.884608,2.241269,NaN
2,2 bedrooms 1 bathroom furnished - Las Letras -...,2.789019,5.549134,2.247958,NaN
3,4C Bravo Murillo,3.123281,3.725543,2.247958,NaN
4,5 Calle Álamo,3.036528,5.195887,2.214951,NaN


In [97]:
# Compute unique word count per hotel (count of columns with values > 0)
hotel_tfidf_stats["unique_tfidf_word_cnt"] = hotel_tfidf_stats.filter(like='_mean').gt(0).sum(axis=1)

# Remove any existing columns before merging to avoid duplicates
df = df.drop(columns=['unique_tfidf_word_cnt'], errors='ignore')

# Merge the unique word count into the original dataframe
df = df.merge(hotel_tfidf_stats[['hotel_name', 'unique_tfidf_word_cnt']], on="hotel_name", how="left")

# Display only relevant columns
print(df[['hotel_name', 'unique_tfidf_word_cnt']].head())

                                          hotel_name  unique_tfidf_word_cnt
0              #10OCA6 - Apartamento en Vista Alegre                     15
1                           1881 Madrid Ventas Hotel                     14
2  2 bedrooms 1 bathroom furnished - Las Letras -...                     18
3                                   4C Bravo Murillo                      9
4                                      5 Calle Álamo                     24


In [99]:
# Export the DataFrame to CSV
df.to_csv("tfidf_text_features.csv", index=False)